# State AI Policy: source validation

## Summary

The 18 sampled records match LegiScan bill identities. Seven bills have an intermediate API stage in a closed session, but **all seven already have a source status of `Failed - Adjourned`**. Their official histories support retaining that status.

Four sampled records originally marked Pending have later signing or chaptering evidence. These are dated validation observations, not replacement training labels. The original and prepared datasets remain unchanged by this notebook.

## Context and method

This diagnostic sample contains six records each from California, Colorado and Texas. Selection first sought three Pending records per state, then filled six using terminal statuses and a deterministic hash order. Texas contributes six terminal-status records. This convenience sample does not estimate a national error rate.

The October 13 observations are preserved as recorded, including their access and coverage limits. The October 19 LegiScan comparison adds matched state, bill and session identifiers, provider stages and history dates. Official state histories and session-closure sources were rechecked October 19 for the seven closed-session cases.

LegiScan's `Introduced`, `Engrossed` and `Passed` labels describe provider stages. A stage, a session closure and an enacted outcome are different facts. The checks below keep them separate. No API key, network access or data regeneration is needed to run this notebook.

## Data

### 1. Load the frozen source and validation tables

Run from the repository root or `notebooks/`. `source_row` is the one-based position in the unchanged raw CSV, not a model feature.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data/raw/Final_Database.csv").exists():
    project_root = project_root.parent

raw_path = project_root / "data/raw/Final_Database.csv"
raw = pd.read_csv(raw_path, dtype=str, keep_default_na=False, skip_blank_lines=False)
validation_dir = project_root / "data/validation"
observations = pd.read_csv(validation_dir / "bill_status_observations.csv", dtype=str, keep_default_na=False)
comparison = pd.read_csv(validation_dir / "legiscan_comparison.csv", dtype=str, keep_default_na=False)
closed = pd.read_csv(validation_dir / "closed_session_review.csv", dtype=str, keep_default_na=False)

### 2. Confirm source linkage and matched identities

The checksum identifies the published snapshot. Matching the source fields prevents a reordered or different CSV from silently changing the records under review.

In [2]:
source_md5 = hashlib.md5(raw_path.read_bytes()).hexdigest()
assert source_md5 == "397a13c53e719e64e52725e7995fd8ca"
assert raw.shape == (2571, 29)
assert len(comparison) == 18 and comparison["source_row"].is_unique
assert comparison["source_md5"].eq(source_md5).all()
assert comparison["identity_matched"].eq("true").all()
assert comparison["api_bill_id"].is_unique

source_columns = ["Jurisdiction", "Bill_Number", "Bill_Title", "Introduction_Date", "Version_Date", "Bill_Status"]
positions = observations["source_row"].astype(int) - 1
pd.testing.assert_frame_equal(
    raw.loc[positions, source_columns].reset_index(drop=True),
    observations[source_columns].reset_index(drop=True),
)
assert comparison.set_index("source_row")["source_status"].to_dict() == observations.set_index("source_row")["Bill_Status"].to_dict()
display(comparison.groupby("state").agg(Records=("source_row", "size"), Matched=("identity_matched", lambda values: values.eq("true").sum())))

,Records,Matched
state,,
CA,6,6
CO,6,6
TX,6,6


## Results

### 3. Compare source statuses with API stages

Read this as a comparison of different fields and check dates. A different stage label alone does not establish an incorrect source outcome. The CSV retains exact bill/session identifiers, public links and API check times.

In [3]:
display(pd.crosstab(
    comparison["source_status"], comparison["api_stage"],
    rownames=["Source status"], colnames=["API stage"],
))

API stage,Engrossed,Failed,Introduced,Passed,Vetoed
Source status,,,,,
Enacted - Act No. 2025-341,0,0,0,1,0
Enacted - Act No. 2025-466,0,0,0,1,0
Enacted - Act No. 2025-673,0,0,0,1,0
Failed - Adjourned,2,1,5,0,0
Failed - Adjourned - Postponed Indefinitely,0,1,0,0,0
Pending,0,0,1,4,1


### 4. Review the seven closed-session cases

Colorado HB25B-1008 belongs to the 2025 extraordinary session, not a regular-session bill with the same number. The official page records Lost; the session adjourned August 26, 2025.

Texas's 89th Regular Session ended June 2, 2025. The six cited histories show incomplete bills. The official Legislative Reference Library explains that unfinished bills die at session end. For Texas, `Failed at session close` is an inference from those histories and that rule, rather than a literal API label.

All seven source records already say `Failed - Adjourned`. Retain the original baseline statuses. Keep API stages and closure evidence in the validation tables; no correction or exclusion is supported by this stage distinction alone. A latest action date is not a failure date.

In [4]:
assert len(closed) == 7
assert closed["source_status"].eq("Failed - Adjourned").all()
assert set(closed["source_row"]) == set(comparison.loc[
    comparison["session_sine_die"].eq("1") & comparison["api_stage"].isin(["Introduced", "Engrossed", "Enrolled"]),
    "source_row",
])
display(closed[["state", "bill", "source_status", "api_stage", "last_action_date", "session_closed_date"]].rename(columns={
    "source_status": "Source status", "api_stage": "API stage",
    "last_action_date": "Last action", "session_closed_date": "Session closed",
}))

,state,bill,Source status,API stage,Last action,Session closed
0,CO,HB25B-1008,Failed - Adjourned,Introduced,2025-08-25,2025-08-26
1,TX,HB641,Failed - Adjourned,Introduced,2025-03-04,2025-06-02
2,TX,SB747,Failed - Adjourned,Engrossed,2025-05-26,2025-06-02
3,TX,SB1298,Failed - Adjourned,Introduced,2025-02-28,2025-06-02
4,TX,HB4448,Failed - Adjourned,Introduced,2025-04-02,2025-06-02
5,TX,SB1822,Failed - Adjourned,Engrossed,2025-05-07,2025-06-02
6,TX,HB1405,Failed - Adjourned,Introduced,2025-05-14,2025-06-02


### 5. Keep four later enactment observations separate

The following source-Pending records have explicit signing or chaptering history evidence. LegiScan code 4 is named `Passed`; the action evidence supplies the additional context. These differences were already observed in the October 13 check and are confirmed by the October 19 API histories.

An updated-label experiment would need a documented cutoff and separate results. It must not introduce later information into features intended to describe an earlier prediction point.

In [5]:
pending_with_evidence = comparison.loc[
    comparison["source_status"].eq("Pending") & comparison["signing_or_chaptering_evidence"].eq("True"),
    ["state", "bill", "source_status", "api_stage", "api_last_action_date", "api_last_action"],
]
assert len(pending_with_evidence) == 4
with pd.option_context("display.max_colwidth", 75):
    display(pending_with_evidence.rename(columns={"api_last_action_date": "Action date", "api_last_action": "Latest history action"}))

,state,bill,source_status,api_stage,Action date,Latest history action
0,CA,SB241,Pending,Passed,2025-10-01,"Chaptered by Secretary of State. Chapter 214, Statutes of 2025."
1,CA,SB951,Pending,Passed,2026-09-30,"Chaptered by Secretary of State. Chapter 860, Statutes of 2026."
7,CO,HB1195,Pending,Passed,2026-06-03,Governor Signed
8,CO,HB1263,Pending,Passed,2026-05-29,Governor Signed


### 6. Preserve the unresolved cases and date definitions

- **California AB2291:** the available history records a canceled hearing. That does not establish a terminal outcome. The API reports `sine_die=0`; no independently refreshed session calendar is used to decide the outcome here.
- **Colorado HB26-1210:** API Vetoed agrees with the earlier veto-history observation. The October 13 official-page Signed Act/summary wording conflict remains documented in `bill_status_observations.csv`.
- **Texas HB4448:** the source short title differs from the official school-library caption. The raw text header identifies Hickland's HB4448, supporting identity. Full-text/version equivalence has not been established; the title difference alone does not justify exclusion.

Three Texas source introduction dates differ from API first-history dates, which record filing. Preserve both definitions rather than overwriting the source introduction dates.

In [6]:
source_dates = pd.to_datetime(observations["Introduction_Date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
date_review = comparison.merge(
    observations[["source_row", "Introduction_Date"]].assign(source_date_iso=source_dates),
    on="source_row", validate="one_to_one",
)
date_review = date_review.loc[date_review["source_date_iso"].ne(date_review["api_first_history_date"])]
display(date_review[["state", "bill", "Introduction_Date", "api_first_history_date"]].rename(columns={
    "Introduction_Date": "Source introduction", "api_first_history_date": "API first history",
}))

,state,bill,Source introduction,API first history
12,TX,HB641,01/14/2025,2024-11-12
13,TX,SB747,01/14/2025,2025-01-08
17,TX,HB1405,01/14/2025,2024-11-19


## Takeaways

The seven closed-session records support the existing Failed/Adjourned baseline labels. Continue with the original-source baseline while keeping later evidence in a separate validation layer. The supervised analysis determines target definitions, eligibility and whether a later, separately reported label-refresh experiment is worthwhile. Topic analysis keeps its own text and feature decisions.

These 18 checks establish sample linkage and specific review findings. They do not validate every corpus record, establish full-text equivalence, or resolve every version/timing issue. This notebook reads files and displays results; it does not write modeling inputs or call the API.

## Sources

- Bhargava and Soliman, [U.S. State AI Legislation Corpus v4](https://zenodo.org/records/20781731), `Final_Database_10.csv`. The repository copy is unchanged.
- [LegiScan](https://legiscan.com/) API bill metadata/history, retrieved October 19, 2026. Selected fields are reorganized and renamed in `legiscan_comparison.csv`; stage labels follow the [API manual, Status/Progress section](https://api.legiscan.com/dl/LegiScan_API_User_Manual.pdf). LegiScan data is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Official state bill-history links and per-record check times appear in the validation CSVs. The seven-bill review also cites the [Texas post-session FAQ](https://www.lrl.texas.gov/whatsNew/client/index.cfm/2025/6/3/Whats-Next-PostSession-FAQ-and-Bill-Statistics-89th-RS) and [Colorado extraordinary-session journal](https://content.leg.colorado.gov/sites/default/files/2025_house_journal.pdf), printed page 76 (PDF page 80).